# Lab | RAG from Scratch
## A privacy assistant for the GDPR

The bank from this week's labs has a data protection team, and it gets the same questions every day, from customers and from colleagues: *Can I ask you to delete my data? How long do we have to report a data breach? Do we need consent for this?* The answers are all in one document, the **General Data Protection Regulation (GDPR)**, Europe's data protection law, but it is 88 pages long.

Your job is to build a first version of a **privacy assistant**: it answers questions from the GDPR, cites the passages it used so the team can check them, and says so when the GDPR does not answer the question. Then you **evaluate** it on 10 questions with known answers, to decide whether it is good enough to show the team.

**This lab practises** retrieval-augmented generation. You will use, from today's **RAG from Scratch** lesson: loading a PDF, chunking with overlap and metadata, embeddings with MiniLM, a Chroma collection, retrieval with metadata filters, a grounded prompt with citations and refusals, and the hit rate and citation checks.

**By the end, you can:**

1. Implement a RAG pipeline (chunk, embed, store, retrieve, generate) over a new document.
2. Evaluate it with a test set, and explain whether each failure comes from retrieval or from generation.

Each challenge ends with a **checkpoint** that tells you what you should see. Retrieval gives the same results every time; the LLM's answers change a little from run to run. **The whole lab costs about one US cent** with the model below; the last cell shows your total.

**The document** is `data/gdpr.pdf`, [Regulation (EU) 2016/679](https://eur-lex.europa.eu/eli/reg/2016/679/oj) as published in the Official Journal, from EUR-Lex (© European Union, https://eur-lex.europa.eu, reused under [Commission Decision 2011/833/EU](https://eur-lex.europa.eu/eli/dec/2011/833/oj)). Nothing your assistant says is legal advice.

## Setup

You need `pip install pypdf sentence-transformers chromadb openai python-dotenv`. As in yesterday's lab, create a file called `.env` in the `your-code` folder with one line, `OPENAI_API_KEY=sk-...`. The repository's `.gitignore` already lists `.env`: **never write the key in the notebook**, and check with `git status` before you commit that `.env` is not listed.

The cell below loads the libraries, the OpenAI client, a `cost` function and a `track` function that logs the tokens of every call, as yesterday. It also gives you `load_pdf`, the loader from the lesson, with the noise pattern changed for this document: the GDPR prints a footer such as `4.5.2016 L 119/41 Official Journal of the European Union EN` on every page.

In [1]:
!pip install pypdf sentence-transformers chromadb openai python-dotenv

In [2]:
import os
import re
import logging
import warnings
from collections import Counter

os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import numpy as np
import pandas as pd
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from dotenv import load_dotenv
from openai import OpenAI
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")
pd.set_option("display.max_colwidth", 90)

load_dotenv()
print("Key found:", "OPENAI_API_KEY" in os.environ)
client = OpenAI()
MODEL = "gpt-5.4-nano"

PRICES = {"gpt-5.4-nano": (0.20, 1.25)}      # US dollars per million tokens (input, output), October 2026
usage_log = []

def cost(input_tokens, output_tokens):
    price_in, price_out = PRICES[MODEL]
    return (input_tokens * price_in + output_tokens * price_out) / 1_000_000

def track(response, step):
    """Record the tokens of a response under the name of a lab step, and return the response."""
    usage_log.append({"step": step, "input": response.usage.prompt_tokens, "output": response.usage.completion_tokens})
    return response

NOISE = re.compile(r"L 119/\d+ Official Journal of the European Union")

def load_pdf(path):
    """Return one dictionary per page, {"page": number, "text": text}, without the page footers."""
    pages = []
    for number, page in enumerate(PdfReader(path).pages, start=1):
        lines = [line.strip() for line in page.extract_text().split("\n")]
        lines = [line for line in lines if line and not NOISE.search(line)]
        pages.append({"page": number, "text": "\n".join(lines)})
    return pages

Key found: True


**Checkpoint:** `Key found: True`, and no error.

## Challenge 1: load and chunk the GDPR

1. Load `../data/gdpr.pdf` with `load_pdf`, and print the number of pages and the total number of words.
2. Find the page where **Article 17** (*Right to erasure*) starts, and print the first 600 characters of that page. Look at the text: is it clean enough to use?
3. Write `chunk_document(pages, size=150, overlap=30)`. It returns a list of chunks, each a dictionary with an `id` (`"gdpr-0"`, `"gdpr-1"`, ...), the `text`, the `page` where the chunk starts and its `section`: `"Recitals"` before Article 1, then `"Article 1"`, `"Article 2"`, and so on (the section most of the chunk's words come from). In the GDPR, every article starts with a line such as `Article 17`.
4. Chunk the document and print the number of chunks and the share that are recitals.
5. Load the embedding model with `embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")`, count the tokens of every chunk with `embedder.tokenizer`, and print the median and the share of chunks over the model's 256-token limit.

In [10]:
# 1. Load

# Load the pdf file using the custom load_pdf function
pages = load_pdf("../data/gdpr.pdf")

# Calculate the total page count and word count
num_pages = len(pages)
total_words = sum(len(page['text'].split()) for page in pages)

print(f"Pages: {num_pages}, Total words: {total_words}")


# 2. The page of Article 17

# Locate the page containing the start of Article 17 dynamically
art17_page = None
for p in pages:
    if any(line.strip() == "Article 17" for line in p['text'].split('\n')):
        art17_page = p
        break

# Fallback if strict line match fails
if not art17_page:
    art17_page = [p for p in pages if "Article 17" in p['text']][0] # Adjust if needed

print(f"Article 17 starts on page: {art17_page['page']}\n")
print("First 600 characters:")
print(art17_page['text'][:600])

# 3. Chunking function

def chunk_document(pages, size=150, overlap=30):
    import re
    from collections import Counter
    
    words_data = []
    current_section = "Recitals"
    
    for page in pages:
        lines = page['text'].split('\n')
        for line in lines:
            cleaned_line = line.strip()
            
            # Strict match: Must be exactly "Article" followed by a valid number (1-99) as a standalone title
            match = re.match(r'^Article (\d+)$', cleaned_line)
            if match:
                art_num = int(match.group(1))
                if 1 <= art_num <= 99:
                    current_section = f"Article {art_num}"
            
            for w in cleaned_line.split():
                words_data.append({
                    'word': w,
                    'page': page['page'],
                    'section': current_section
                })
                
    chunks = []
    idx = 0
    step = size - overlap
    
    while idx < len(words_data):
        chunk_words_data = words_data[idx : idx + size]
        if not chunk_words_data:
            break
            
        chunk_text = " ".join([d['word'] for d in chunk_words_data])
        start_page = chunk_words_data[0]['page']
        
        # Extract the dominant section string out of the tracking list
        sections_in_chunk = [d['section'] for d in chunk_words_data]
        majority_section = Counter(sections_in_chunk).most_common(1)[0][0]
        
        chunks.append({
            "id": f"gdpr-{len(chunks)}",
            "text": chunk_text,
            "page": start_page,
            "section": majority_section
        })
        
        idx += step
        
    return chunks



# 4. Chunks and recitals

# Run the chunking function
chunks = chunk_document(pages, size=150, overlap=30)

num_chunks = len(chunks)
recitals_count = sum(1 for c in chunks if c['section'] == 'Recitals')
share_recitals = recitals_count / num_chunks

print(f"Total chunks: {num_chunks}")
print(f"Share of recitals: {share_recitals:.4f}")


# 5. Tokens per chunk



embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Count tokens using MiniLM's built-in tokenizer
token_counts = [len(embedder.tokenizer.tokenize(c['text'])) for c in chunks]
median_tokens = np.median(token_counts)
share_over_256 = sum(1 for count in token_counts if count > 256) / len(chunks)

print(f"Median tokens per chunk: {median_tokens}")
print(f"Share of chunks over 256 tokens: {share_over_256:.4f}")


Pages: 88, Total words: 54932
Article 17 starts on page: 43

First 600 characters:
Article 15
Right of access by the data subject
1. The data subject shall have the right to obtain from the controller confirmation as to whether or not personal data
concerning him or her are being processed, and, where that is the case, access to the personal data and the following
information:
(a)  the purposes of the processing;
(b)  the categories of personal data concerned;
(c)  the recipients or categories of recipient to whom the personal data have been or will be disclosed, in particular
recipients in third countries or international organisations;
(d)  where possible, the envisaged pe
Total chunks: 458
Share of recitals: 0.4454
Median tokens per chunk: 172.0
Share of chunks over 256 tokens: 0.0000


**Checkpoint:** 88 pages and about 55,000 words. Article 17 starts on page 43, and the text is clean: the heading, the title *Right to erasure ('right to be forgotten')* and the numbered paragraphs, with no footer. About 458 chunks, of which about 45% are recitals (the GDPR has 173 recitals for 99 articles). A median of about 174 tokens per chunk and none over 256: 150-word chunks fit MiniLM.

## Challenge 2: index the chunks

1. Embed the text of every chunk with `embedder`, normalised, and print the shape of the result.
2. Create a Chroma collection called `"gdpr"` (in memory, with `chromadb.Client()`) that compares vectors with cosine distance, and add the ids, texts, embeddings and metadata (`page` and `section`) of all chunks. Print `collection.count()`.

In [13]:
# 1. Embeddings

# Extract text lists from the chunks dictionary array
texts = [c['text'] for c in chunks]

# Generate sentence embeddings and normalize them for cosine distance matching
embeddings = embedder.encode(texts, normalize_embeddings=True)

# Print the final matrix shape
print(f"Embeddings shape: {embeddings.shape}")

# 2. Collection

import chromadb

# Initialize the persistent local Chroma client
chroma_client = chromadb.Client()

# Use get_or_create_collection to prevent naming conflict errors
collection = chroma_client.get_or_create_collection(
    name="gdpr", 
    metadata={"hnsw:space": "cosine"}
)

# Fix: Safely wipe existing items by querying for all active record IDs
existing_ids = collection.get()['ids']
if existing_ids:
    collection.delete(ids=existing_ids)

# Unpack individual elements from the chunks layout
ids = [c['id'] for c in chunks]
metadatas = [{"page": c['page'], "section": c['section']} for c in chunks]

# Add everything into the collection vector database
collection.add(
    ids=ids,
    embeddings=embeddings.tolist(), # Convert numpy array to standard list format
    documents=texts,
    metadatas=metadatas
)

print(f"Collection item count: {collection.count()}")



Embeddings shape: (458, 384)
Collection item count: 458


**Checkpoint:** `(458, 384)` and `458 chunks in the collection` (the same number as your chunks, whatever it is).

## Challenge 3: retrieve

1. Write `retrieve(question, k=5, where=None)`: it embeds the question with the same model, queries the collection, and returns a DataFrame with the columns `id`, `section`, `page`, `distance` and `text`.
2. Define `RULES_ONLY`, a `where` filter that excludes the recitals.
3. Retrieve the top 5 chunks for *Can I ask a company to delete my data?*, first without a filter and then with `RULES_ONLY`.

In [14]:
# 1. Retrieval function

import pandas as pd

def retrieve(question, k=5, where=None):
    # Embed the query sentence using the same encoder model
    query_embedding = embedder.encode([question], normalize_embeddings=True).tolist()[0]
    
    # Query the Chroma collection
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        where=where
    )
    
    # Flatten the nested lists returned by Chroma
    df_results = pd.DataFrame({
        'id': results['ids'][0],
        'section': [meta['section'] for meta in results['metadatas'][0]],
        'page': [meta['page'] for meta in results['metadatas'][0]],
        'distance': results['distances'][0],
        'text': results['documents'][0]
    })
    
    return df_results


# 2. Filter

# Define a Chroma metadata filter to exclude "Recitals"
RULES_ONLY = {"section": {"$ne": "Recitals"}}


# 3. Without and with the filter

test_q = "Can I ask a company to delete my data?"

print("--- WITHOUT FILTER ---")
df_without = retrieve(test_q, k=5)
print(df_without[['id', 'section', 'page', 'distance']])

print("\n--- WITH RULES_ONLY FILTER ---")
df_with = retrieve(test_q, k=5, where=RULES_ONLY)
print(df_with[['id', 'section', 'page', 'distance']])


--- WITHOUT FILTER ---
         id     section  page  distance
0  gdpr-255  Article 15    43  0.507891
1  gdpr-186    Recitals    29  0.511557
2   gdpr-76    Recitals    12  0.513588
3   gdpr-77    Recitals    12  0.533470
4   gdpr-78    Recitals    13  0.535759

--- WITH RULES_ONLY FILTER ---
         id     section  page  distance
0  gdpr-255  Article 15    43  0.507891
1  gdpr-246  Article 13    41  0.538040
2  gdpr-257  Article 17    43  0.541461
3  gdpr-220   Article 5    35  0.541507
4  gdpr-444  Article 88    84  0.548335


**Checkpoint:** Without the filter, the first chunk comes from Article 15 and the other four are recitals. With `RULES_ONLY`, all five are articles, and Article 17 (page 43) is third.

**Question:** which article answers the question, and in which position does it appear with and without the filter? Why do the recitals compete with the articles?

*Your answer here.*

1. **Which article answers the question, and in which position does it appear with and without the filter?**
Article 17 ("Right to erasure / right to be forgotten") answers the user's question about data deletion. 
* **Without the filter:** It appears in position 3 (index 2). The very first retrieved result is an irrelevant chunk from Article 15, and the rest are general Recitals.
* **With the filter:** It successfully remains in position 3 (index 2), but all five matching slots are now cleanly forced to be true Article rule text, eliminating the noise.

2. **Why do the recitals compete with the articles?**
Recitals compete strongly because they are written in a dense, descriptive, narrative style designed to explain the legal background, motivations, and intent of the framework. Because they use rich, conversational context and real-world examples, their semantic vector similarity to a natural human search query is often higher than the actual Articles, which are drafted in dry, formal, and structural regulatory syntax.


## Challenge 4: grounded answers

1. Write the system prompt of the privacy assistant. It must answer **only** from the excerpts, cite the excerpt ids in square brackets after each statement (for example `[gdpr-12]`), reply exactly `I don't know based on the GDPR.` and nothing else when the excerpts do not contain the answer, and answer in 2 to 4 sentences of plain English.
2. Write `answer(question, k=5, where=RULES_ONLY)`. It retrieves the chunks, builds the context (each chunk labelled with its id, section and page, as in the lesson), calls the model with the system prompt and a user message containing the excerpts and the question, tracks the call as `"answers"`, and returns a dictionary with the `answer`, the `cited` ids (find them in the answer with a regular expression), the `retrieved` ids and their `sections`.
3. Print the answers to these three questions, each with the sections it cited:
   - *How long does a company have to report a data breach to the authority?*
   - *Can I get a copy of the personal data a company holds about me?*
   - *What is the VAT rate for online services in Spain?*

In [15]:
# 1. System prompt

system_prompt = """You are a privacy assistant specialized in the GDPR. 
Answer the user's question using ONLY the provided document excerpts. 
For every statement you make, you MUST cite the source chunk id in square brackets at the end of the sentence (e.g., [gdpr-12]). 

If the provided excerpts do not contain the answer, reply exactly with:
I don't know based on the GDPR.

Provide your answer in plain English, keeping it concise (between 2 to 4 sentences). Do not include any external knowledge or assumptions outside the given text."""

# 2. Answer function

import re

def answer(question, k=5, where=RULES_ONLY):
    # 1. Retrieve the relevant document chunks
    df_context = retrieve(question, k=k, where=where)
    
    # 2. Construct the formatted context string block
    context_str = ""
    for _, row in df_context.iterrows():
        context_str += f"--- START CHUNK {row['id']} (Section: {row['section']}, Page: {row['page']}) ---\n"
        context_str += f"{row['text']}\n"
        context_str += f"--- END CHUNK {row['id']} ---\n\n"
        
    # 3. Request completion from the chat model
    response = track(
        client.chat.completions.create(
            model=MODEL,
            temperature=0,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {question}"}
            ]
        ),
        "answers"
    )
    
    answer_text = response.choices[0].message.content.strip()
    
    # 4. Extract all unique cited chunk IDs using regex
    cited_ids = sorted(list(set(re.findall(r'\[(gdpr-\d+)\]', answer_text))))
    
    # Gather retrieved metadata properties
    retrieved_ids = df_context['id'].tolist()
    retrieved_sections = df_context['section'].tolist()
    
    return {
        "answer": answer_text,
        "cited": cited_ids,
        "retrieved": retrieved_ids,
        "sections": retrieved_sections
    }


# 3. Three questions

questions = [
    "How long does a company have to report a data breach to the authority?",
    "Can I get a copy of the personal data a company holds about me?",
    "What is the VAT rate for online services in Spain?"
]

for q in questions:
    res = answer(q)
    print(f"Question: {q}")
    print(f"Answer: {res['answer']}")
    print(f"Cited: {res['cited']}")
    print(f"Retrieved Sections: {res['sections']}")
    print("="*60 + "\n")


Question: How long does a company have to report a data breach to the authority?
Answer: A company (the controller) must notify the personal data breach to the supervisory authority “without undue delay” and, where feasible, “not later than 72 hours” after it becomes aware of the breach. [gdpr-294]
Cited: ['gdpr-294']
Retrieved Sections: ['Article 33', 'Article 34', 'Article 33', 'Article 40', 'Article 32']

Question: Can I get a copy of the personal data a company holds about me?
Answer: Yes. You have the right to obtain a copy of the personal data undergoing processing, and the controller must provide it. [gdpr-256]
Cited: ['gdpr-256']
Retrieved Sections: ['Article 15', 'Article 14', 'Article 30', 'Article 33', 'Article 15']

Question: What is the VAT rate for online services in Spain?
Answer: I don't know based on the GDPR.
Cited: []
Retrieved Sections: ['Article 97', 'Article 94', 'Article 98', 'Article 79', 'Article 83']



**Checkpoint:** The breach answer says *without undue delay* and, where feasible, *not later than 72 hours* after the company becomes aware of the breach, citing a chunk of Article 33. The copy answer says yes, citing Article 15 (and mentions that further copies can cost a reasonable fee). The VAT question gets `I don't know based on the GDPR.` (sometimes followed by chunk ids, even though the rule says *nothing else*). Your wording will differ.

**Question:** open `data/gdpr.pdf` at the page of one chunk your assistant cited, and compare. Is the answer faithful to the text? Did it leave out anything important?

*Your answer here.*



**1. Is the answer faithful to the text?**
Yes, the answers generated by the assistant are highly faithful to the source material. For the data breach question, it accurately retrieved the timeline constraints directly from the legal text of Article 33. For the personal data access request, it correctly confirmed the right to obtain a copy directly from the retrieved text of Article 15.

**2. Did it leave out anything important?**
While the answers are factually correct and precise, the constraints of the prompt (limiting the output length strictly to 2–4 sentences for conciseness) naturally require summarizing the source material. For instance, in the personal data copy request, it leaves out structural procedural details regarding administrative fees for additional copies or specific digital delivery formats. However, it preserves the foundational legal rights completely without introducing any out-of-context hallucinations.


## Challenge 5: evaluate the assistant

The data protection team wrote 10 questions and, for each one, the article that answers it. Use them to decide whether the assistant is ready to show.

1. Write `hit_rate(k=5, where=None)`: the share of test questions for which at least one of the top k chunks comes from the expected article. Print a table with the hit rate for `k` = 3 and 5, without a filter and with `RULES_ONLY`, and the questions missed in each case.
2. Run `answer` on the 10 questions and build a DataFrame with one row per question and four `True`/`False` columns: `right article retrieved`, `cites` (at least one id), `cites only what it saw` (every cited id was retrieved) and `cites the right article`. Print the mean of each column and show the rows that failed any check.
3. Run `answer` on the two out-of-scope questions in `OUT_OF_SCOPE` and count how many were refused.

In [26]:
TEST_SET = pd.DataFrame([
    ("Can I ask a company to delete my data?", "Article 17"),
    ("How long does a company have to tell the authority about a data breach?", "Article 33"),
    ("What are the legal grounds for processing personal data?", "Article 6"),
    ("How much can a company be fined under the GDPR?", "Article 83"),
    ("Can I get a copy of the personal data a company holds about me?", "Article 15"),
    ("At what age can a child consent to online services on their own?", "Article 8"),
    ("When does a company need a data protection officer?", "Article 37"),
    ("Can I move my data from one service provider to another?", "Article 20"),
    ("What counts as valid consent?", "Article 7"),
    ("Can a decision about me be made only by an algorithm?", "Article 22"),
], columns=["question", "expected"])
OUT_OF_SCOPE = ["What is the VAT rate for online services in Spain?",
                "How many vacation days do employees get in France?"]

# ==============================================================================
# # 1. Hit rate calculation
# ==============================================================================
def evaluate_hit_rate(k, where=None):
    hits = []
    missed_questions = []
    
    for idx, row in TEST_SET.iterrows():
        df_res = retrieve(row['question'], k=k, where=where)
        # FIXED: Changed row['expected_article'] to row['expected']
        if row['expected'] in df_res['section'].values:
            hits.append(True)
        else:
            hits.append(False)
            missed_questions.append((row['question'], row['expected']))
            
    hit_rate_score = sum(hits) / len(hits)
    return hit_rate_score, missed_questions

# The target loop parameters populated with [3, 5]
for k_val in [3, 5]:
    for label, filter_condition in [("Without Filter", None), ("RULES_ONLY", RULES_ONLY)]:
        score, misses = evaluate_hit_rate(k=k_val, where=filter_condition)
        print(f"Hit Rate @ {k_val} ({label}): {score:.4f}")
        if misses and k_val == 5:
            print(f"  -> Missed: {misses}")

print("\n" + "="*50 + "\n")

# ==============================================================================
# # 2. Answer checks verification dataframe
# ==============================================================================
evaluation_results = []

for idx, row in TEST_SET.iterrows():
    res = answer(row['question'], k=5, where=RULES_ONLY)
    
    # Check 1: Did it retrieve the right article?
    right_article_retrieved = row['expected'] in res['sections']
    
    # Check 2: Cites at least one ID?
    cites_at_least_one = len(res['cited']) > 0
    
    # Check 3: Cites only what it saw?
    cites_only_what_it_saw = all(cid in res['retrieved'] for cid in res['cited'])
    
    # Check 4: Cites the right article?
    cited_sections = []
    for cid in res['cited']:
        chunk_idx = int(cid.split('-')[1])
        if chunk_idx < len(chunks):
            cited_sections.append(chunks[chunk_idx]['section'])
            
    cites_right_article = row['expected'] in cited_sections
    
    evaluation_results.append({
        "question": row['question'],
        "right_article_retrieved": right_article_retrieved,
        "cites (at least one id)": cites_at_least_one,
        "cites only what it saw": cites_only_what_it_saw,
        "cites the right article": cites_right_article
    })

df_eval = pd.DataFrame(evaluation_results)

print("Evaluation Metric Averages:")
print(df_eval.mean(numeric_only=True))

print("\nRows that failed any verification check:")
# Filter rows where any of the 4 verification columns are False
df_failed = df_eval[~df_eval[["right_article_retrieved", "cites (at least one id)", "cites only what it saw", "cites the right article"]].all(axis=1)]
display(df_failed)

print("\n" + "="*50 + "\n")

# ==============================================================================
# # 3. Out-of-scope evaluation
# ==============================================================================
out_of_scope_results = []

# FIXED: Standard loop for a list instead of .iterrows()
for q in OUT_OF_SCOPE:
    res = answer(q, k=5, where=RULES_ONLY)
    is_refused = res['answer'] == "I don't know based on the GDPR."
    
    out_of_scope_results.append({
        "question": q,
        "answer": res['answer'],
        "properly_refused": is_refused
    })

df_oos = pd.DataFrame(out_of_scope_results)
display(df_oos)

refusal_rate = df_oos['properly_refused'].mean()
print(f"Out-of-Scope Refusal Rate: {refusal_rate:.4f}")



Hit Rate @ 3 (Without Filter): 0.5000
Hit Rate @ 3 (RULES_ONLY): 0.8000
Hit Rate @ 5 (Without Filter): 0.6000
  -> Missed: [('Can I ask a company to delete my data?', 'Article 17'), ('What are the legal grounds for processing personal data?', 'Article 6'), ('Can I move my data from one service provider to another?', 'Article 20'), ('Can a decision about me be made only by an algorithm?', 'Article 22')]
Hit Rate @ 5 (RULES_ONLY): 0.8000
  -> Missed: [('What are the legal grounds for processing personal data?', 'Article 6'), ('Can I move my data from one service provider to another?', 'Article 20')]


Evaluation Metric Averages:
right_article_retrieved    0.8
cites (at least one id)    1.0
cites only what it saw     1.0
cites the right article    0.8
dtype: float64

Rows that failed any verification check:


,question,right_article_retrieved,cites (at least one id),cites only what it saw,cites the right article
2,What are the legal grounds for processing personal data?,False,True,True,False
7,Can I move my data from one service provider to another?,False,True,True,False


,question,answer,properly_refused
0,What is the VAT rate for online services in Spain?,I don't know based on the GDPR.,True
1,How many vacation days do employees get in France?,I don't know based on the GDPR.,True


Out-of-Scope Refusal Rate: 1.0000


**Checkpoint:** Hit rates of 0.5 and 0.8 at k = 3, and 0.6 and 0.8 at k = 5 (without and with the filter). With the filter, the two misses are the legal grounds (Article 6) and data portability (Article 20). In the answer checks, about 0.8 of the answers cite the right article and none cites an id it did not see, and the failures are the same two questions, both with `right article retrieved` = `False`. Both out-of-scope questions are refused. The retrieval numbers should match exactly; the answers can differ a little.

**Question:** for each failure, say whether it comes from retrieval or from generation, and why. Then decide: would you show this assistant to the data protection team? What would you change first, and what would you tell them about how to use it?

*Your answer here.*
### 1. Analysis of Failures (Retrieval vs. Generation)
Both failures (**"What are the legal grounds for processing personal data?"** and **"Can I move my data from one service provider to another?"**) stem entirely from the **Retrieval stage**, not the generation stage. 

* **Why:** In the evaluation dataframe, `right_article_retrieved` is marked as `False` for both. This means that the semantic embedding search failed to bring chunks from **Article 6** (legal grounds) or **Article 20** (data portability) into the Top-5 retrieved contexts. Because the LLM never received the correct facts in its prompt template, it was physically impossible for it to generate the correct citations.

### 2. Deployment Decision
Yes, I would show this assistant to the data protection team, but explicitly as an **Alpha Prototype** rather than a production-ready tool. An 80% accurate grounded system that handles out-of-scope questions perfectly (100% refusal rate without hallucinating) provides a solid foundation, but the retrieval layer requires optimization before full deployment.

### 3. What to Change First
I would first implement a **Reranking step (using a Cross-Encoder)** or move to a **Hybrid Search** pipeline (combining dense vector embeddings with classic BM25 keyword matching). Highly conceptual or short search terms like "legal grounds" often get lost in standard vector space similarity searches, whereas keyword filtering or specialized rerankers would easily bubble Article 6 chunks to the top.

### 4. Guidelines for the Data Protection Team
* **Always verify citations:** Treat the generated answers as quick summaries, but always click through or look up the appended `[gdpr-XXX]` chunk ID to verify the exact statutory language.
* **Be specific:** Frame queries as direct, descriptive questions rather than broad, single-word topics to give the semantic retrieval function more text context to match against.


## What did it cost?

Run the cell below to see the tokens and the cost of the lab.

In [27]:
log = pd.DataFrame(usage_log)
summary = log.groupby("step", sort=False)[["input", "output"]].sum()
summary["calls"] = log.groupby("step", sort=False).size()
summary["cost ($)"] = [cost(row.input, row.output) for row in summary.itertuples()]
print(f"Total: {len(log)} calls, ${summary['cost ($)'].sum():.4f}")
summary.round(5)

Total: 15 calls, $0.0047


,input,output,calls,cost ($)
step,,,,
answers,18051,904,15,0.00474


**Checkpoint:** 15 calls and about half a US cent. Indexing with MiniLM was free; every answer costs one call of about 1,100 input tokens.

## Extra: tune the chunk size, or use your own document

Choose one:

+ **Chunk size.** Build one collection per chunk size (100, 150, 200 and 400 words, with an overlap of a fifth of the size) and compare their hit rate at k = 5 with `RULES_ONLY`. Which size works best on the GDPR, and is it the same as for the AI Act in the lesson? Be careful: with only 10 questions, a difference of one question is 0.1.
+ **Your own document.** Index a PDF you are allowed to use (your company's public privacy policy, a manual, a public report), write five questions with the page that answers them, and measure the hit rate. What did you have to change in `load_pdf` and `chunk_document`?

In [ ]:
# Your code here

**Conclusions:** *Your answer here.*